In [1]:
'''TODO EARLY STOPPING, SELF/CROSS ATTENTION IN AUTO ENCODER, BLOCKING ON ALGS OR PASSWORDS OR SOURCES, REGULARIZATION ON LAYERS AND/OR SEPERATE NORMALIZATION LAYERS'''
from ollama import chat
from ollama import ChatResponse
import secrets
from Crypto.Cipher import AES
from sklearn.cluster import SpectralClustering
from sklearn.model_selection import train_test_split
from Crypto.Cipher import AES
from Crypto.Hash import SHAKE128
from keras.layers import Input, Dense, Dropout, Flatten, Conv2D, MaxPool2D, Attention, Normalization, Reshape, Conv2DTranspose, MultiHeadAttention, BatchNormalization, Activation
import polars as pl
import os
from Crypto.Util.Padding import pad, unpad
import math
from keras import Model, Input, Sequential
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import plotly.express as px
from keras.callbacks import EarlyStopping

2025-04-14 16:38:40.723670: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-14 16:38:40.739384: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-14 16:38:40.744009: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-14 16:38:40.755212: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available: 1


I0000 00:00:1744670322.982175    8426 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1744670323.024883    8426 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1744670323.025185    8426 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355


In [3]:
'''Passwords & Hashes'''
passwords = [b"Andromeda", b"Neptune", b"Jupiter", b"Saturn"]
algs= ['128', '256'] #be sure to add 512 eventually
hash128 = {password: SHAKE128.new(password).read(128 // 8) for password in passwords}
hash256 = {password: SHAKE128.new(password).read(256 // 8) for password in passwords}
hash512 = {password: SHAKE128.new(password).read(512 // 8) for password in passwords}
print(hash512[b'Andromeda'])
print(hash128)

b"p\x17Y{\xbc\x82\x06\xb7\tr\xd7\xeb)~\xf4\x00%\xf0R\xf73NF\xb3\x86\xd2\x9dJ\x84\x9b(^\x8c\xc3a\x93\xf2\xca\xc6\xafI\xaf\x93\x1f\xe6\x8d\x05)\xb5\xa1\x08\x86X\x02\xc9'\x07\x84\xf3bl\\\x99f"
{b'Andromeda': b'p\x17Y{\xbc\x82\x06\xb7\tr\xd7\xeb)~\xf4\x00', b'Neptune': b'\xe3\xbc\xf4\x89\xd6\x13I4\xe9\x08&\xfd\xe2a[#', b'Jupiter': b'/Z\xd0\xd26\x95\x9b\xe5u\x9bs\xbcub-\x86', b'Saturn': b'L4 \xab\x95\n\xb2\xffM\x14O-\xf1\xe2a\x14'}


In [4]:
'''Encrypt Messages'''
'''need all ciphertexts to be the same size, maybe just set a block limit based on the shortest message length?'''
#should only need to be done once
def encryptMessages(size, pass_dict, max_blocks):
    max_bytes = max_blocks * 512 // 8 #should be a perfect division anyways
    passwords = list(pass_dict.keys())
    for (root, dirs, files) in os.walk("../HumanReadable"):
        for file in files:
            with open(f'../HumanReadable/{file}', 'rb') as plaintextstore:
                plaintext = plaintextstore.read()
                for password in passwords:
                    #print(len(pass_dict[password]))
                    with open(f'../{size}ciphertext/{size}_{password}_{file}', 'wb') as cipherstore:
                        ciphertext = AES.new(pass_dict[password], AES.MODE_ECB).encrypt(pad(plaintext[:max_bytes], 512, 'iso7816'))
                        cipherstore.write(ciphertext)
#not most blocks present but most blocks to use = num blocks in shortest message
max_blocks = 1000
image_shape = [8, 8]
for (root, dirs, files) in os.walk("../HumanReadable"):
    for file in files:
        with open(f'../HumanReadable/{file}', 'rb') as plaintextstore:
            plaintext = plaintextstore.read()
            if math.ceil((len(plaintext) * 8) / 512) < max_blocks:
                max_blocks = math.ceil((len(plaintext) * 8) / 512)
print(math.sqrt((512 * max_blocks) / 8))
img_size = (512 * max_blocks) / 8
s = math.floor(math.sqrt(img_size))
l = s
w = s
r = img_size - (l*w)
while r != 0:
    l -= 1
    w = math.floor(img_size / l)
    r = img_size - (l*w)
image_shape = [l,w]
image_shape.append(1)
image_shape = tuple(image_shape)
print(max_blocks, image_shape)
assert(image_shape[0] * image_shape[1] == (512 * max_blocks) / 8)
encryptMessages('128', hash128, max_blocks)
encryptMessages('256', hash256, max_blocks)

49.95998398718719
39 (48, 52, 1)


In [5]:
'''Compile Training Data'''
columns = ['alg', 'password', 'source_file', 'plaintext', 'ciphertext']
data = []
for alg in algs:
    for password in passwords:
        for (root, dirs, files) in os.walk("../HumanReadable"):
            for source_file in files:
                with open(f'../HumanReadable/{source_file}', 'rb') as plaintextstore:
                    plaintext = plaintextstore.read()
                    with open(f'../{alg}ciphertext/{alg}_{password}_{source_file}', 'rb') as cipherstore:
                        #If there is an error run encryption again
                        ciphertext = cipherstore.read()
                        #print(len(ciphertext))
                        #normalize and want (l,w,d) shape
                        ciphertextimage = [[[int(byte)/255] for byte in ciphertext[row*image_shape[0] : row*image_shape[0] + image_shape[1]]] for row in range(image_shape[0])]
                        ciphertextimagearray = np.asarray(ciphertextimage, np.float32)
                        data.append([alg, password, source_file, plaintext, ciphertextimagearray])

df = pl.LazyFrame(data, columns, orient='row')
df = df.collect()
df = df.sample(fraction=1, shuffle= True)
#df = df.to_dummies(columns[:-2])
train_size = math.floor( .9 * len(df))
test_size = len(df) - train_size
train, test = df.head(train_size), df.tail(test_size)
train_data, test_data = train['ciphertext'], test['ciphertext']
train_targets, test_targets = train.select(pl.exclude('ciphertext')), test.select(pl.exclude('ciphertext'))

In [6]:
train_data_format = np.array(train_data.to_list())
test_data_format = np.array(test_data.to_list())
sources = df['source_file'].unique()
print(sources)

shape: (200,)
Series: 'source_file' [str]
[
	"Convo1Message1"
	"Convo33Message2"
	"Convo20Message4"
	"Convo10Message2"
	"Convo39Message0"
	…
	"Convo6Message4"
	"Convo29Message1"
	"Convo3Message2"
	"Convo17Message0"
	"Convo16Message3"
]


In [7]:
def make_mapping(data, enum_vals, col_name):
    mapping = {enum_val: i for i, enum_val in enumerate(enum_vals)}
    mapped_data = data.with_columns(
    pl.col(col_name).replace_strict(mapping)
    )
    return mapped_data, mapping

In [8]:
print(train_targets.head())

shape: (5, 4)
┌─────┬──────────────┬─────────────────┬─────────────────────────────────┐
│ alg ┆ password     ┆ source_file     ┆ plaintext                       │
│ --- ┆ ---          ┆ ---             ┆ ---                             │
│ str ┆ binary       ┆ str             ┆ binary                          │
╞═════╪══════════════╪═════════════════╪═════════════════════════════════╡
│ 128 ┆ b"Andromeda" ┆ Convo35Message1 ┆ b"Subject:\x20Nightingale\x20-… │
│ 128 ┆ b"Jupiter"   ┆ Convo5Message4  ┆ b"Subject:\x20Nightingale\x20t… │
│ 256 ┆ b"Saturn"    ┆ Convo14Message0 ┆ b"Subject:\x20Nightingale\x20U… │
│ 128 ┆ b"Saturn"    ┆ Convo30Message4 ┆ b"Subject:\x20Re:\x20Chronos\x… │
│ 128 ┆ b"Saturn"    ┆ Convo33Message1 ┆ b"Subject:\x20Re:\x20Chronos\x… │
└─────┴──────────────┴─────────────────┴─────────────────────────────────┘


In [9]:
'''Autoencoder'''
'''normalization before attention, batch normalization before convolution'''
'''CNN -> batchnormalize -> activation -> dropout'''
'''normalization -> attention -> dropout'''
#is mean pooling worth investigating
#will need to play around with amount of parameters
#should i include more than just the ciphertext?
encoder_input = Input(shape=image_shape)
#4x4 byte matrix is AES operator size seems fitting
#blocks*4 is the amount of AES-128 blocks which seems like a decent starting metric
encoder = Conv2D(max_blocks*4, (4,4), padding='same')(encoder_input)
encoder = BatchNormalization(axis=-1)(encoder)
encoder = Activation('relu')(encoder)
encoder = Dropout(.2)(encoder)
encoder = MaxPool2D(pool_size=(4,4), strides=4)(encoder)
encoder = BatchNormalization(axis=-1)(encoder)

encoder = Conv2D(max_blocks, (4,4), padding='same', activation='relu')(encoder)
encoder = BatchNormalization(axis=-1)(encoder)
encoder = Activation('relu')(encoder)
encoder = Dropout(.2)(encoder)
encoder = MaxPool2D(pool_size=(2,2), strides=4)(encoder)
encoder_attended = Normalization(axis = -1)(encoder)
'''maybe try cross attention?'''
encoder_attention = MultiHeadAttention(num_heads=4, key_dim=16)(encoder_attended, encoder_attended)
encoder = Dropout(.2)(encoder_attention)
encoder = Flatten()(encoder)
embedded_layer = Dense(3, activation='relu')(encoder)
decoder1 = Dense(4, activation='relu')(embedded_layer)
decoder2 = Reshape((2,2,1))(decoder1)
decoder3 = Conv2DTranspose(max_blocks, (2,2), strides = (l // 2,w // 2), padding='same', activation='relu')(decoder2)
#decoder3 = Conv2DTranspose(max_blocks * 4, (2,2), strides = (l // 2,w // 2), padding='same', activation='relu')(decoder2)
decoder4 = Conv2D(1, (4, 4), activation="sigmoid", padding="same")(decoder3)

autoencoder = Model(encoder_input, decoder4)
encoder = Model(encoder_input, embedded_layer)


I0000 00:00:1744670327.456711    8426 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1744670327.457098    8426 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1744670327.457257    8426 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1744670327.522522    8426 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [10]:
autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 48, 52, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 48, 52,    │      2,652 │ input_layer[0][0] │
│                     │ 156)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 48, 52,    │        624 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 156)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 48, 52,    │          0 │ batch_normalizat… │
│ (Activation)        │ 156)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 48, 52,    │          0 │ activation[0][0]  │
│                     │ 156)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 12, 13,    │          0 │ dropout[0][0]     │
│ (MaxPooling2D)      │ 156)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 12, 13,    │        624 │ max_pooling2d[0]… │
│ (BatchNormalizatio… │ 156)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 12, 13,    │     97,383 │ batch_normalizat… │
│                     │ 39)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 12, 13,    │        156 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 39)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 12, 13,    │          0 │ batch_normalizat… │
│ (Activation)        │ 39)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 12, 13,    │          0 │ activation_1[0][… │
│                     │ 39)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 3, 3, 39)  │          0 │ dropout_1[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 3, 3, 39)  │         79 │ max_pooling2d_1[… │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 3, 3, 39)  │     10,215 │ normalization[0]… │
│ (MultiHeadAttentio… │                   │            │ normalization[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 3, 3, 39)  │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 351)       │          0 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 3)         │      1,056 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 4)         │         16 │ dense[0][0]     

 Total params: 113,625 (443.85 KB)

 Trainable params: 112,844 (440.80 KB)

 Non-trainable params: 781 (3.05 KB)

In [ ]:
'''keras example uses binary_crossentropy but they also had binary images, description says its for binary classification though'''
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder_callback = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
history = autoencoder.fit(train_data_format, train_data_format, epochs = 200, callbacks=[autoencoder_callback])

Epoch 1/200


I0000 00:00:1744670331.494772    8513 service.cc:146] XLA service 0x7589c000a510 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1744670331.494812    8513 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce GTX 1070, Compute Capability 6.1
2025-04-14 16:38:51.567218: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-04-14 16:38:51.973902: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90101


12/45 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0838

I0000 00:00:1744670336.927599    8513 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - loss: 0.0839
Epoch 2/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0839
Epoch 3/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0839
Epoch 4/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0839
Epoch 5/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0840
Epoch 6/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0839
Epoch 7/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0838
Epoch 8/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0839
Epoch 9/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0839
Epoch 10/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0838
Epoch 11/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0839
Epoch 12/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0839
Epoch 13/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0838
Epoch 14/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0839
Epoch 15/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0839
E

KeyboardInterrupt: 

In [ ]:
encode = encoder.predict(test_data_format)

In [ ]:
len(train_data.to_list())

In [ ]:
'''Supervised Classification'''
activation = ''
if len(algs) == 2:
    activation = 'sigmoid'
else:
    activation = 'softmax'
# encoder_input = Input(shape=image_shape)
# algclass1 = Conv2D(max_blocks*4, (4,4), padding='same', activation='relu')
# alg_drop1 = Dropout(.3)
# algclass2 = MaxPool2D(pool_size=(4,4), strides=4)
# algclass3 = Conv2D(max_blocks, (4,4), padding='same', activation='relu')
# alg_drop2 = Dropout(.3)
# algclass4 = MaxPool2D(pool_size=(4,4), strides=4)
# algclass5 = Flatten()
# alg_prediction_layer = Dense(len(algs), activation=activation)
# alg_classifier = Sequential([encoder_input, algclass1, alg_drop1, algclass2, algclass3, alg_drop2, algclass4, algclass5, alg_prediction_layer])
alg_heads = 4
alg_key_dim =16
alg_atten_dim = alg_heads*alg_key_dim
alg_input = Input(shape=image_shape)
alg_classifier = Conv2D(max_blocks*4, (4,4), padding='same')(alg_input)
alg_classifier = BatchNormalization(axis = -1)(alg_classifier)
alg_classifier = Activation('relu')(alg_classifier)
alg_classifier = Dropout(.3)(alg_classifier)
alg_classifier = MaxPool2D(pool_size=(4,4), strides=4)(alg_classifier)
alg_classifier = BatchNormalization(axis = -1)(alg_classifier)
alg_classifier = Conv2D(max_blocks, (4,4), padding='same')(alg_classifier)
alg_classifier = BatchNormalization(axis = -1)(alg_classifier)
alg_classifier = Activation('relu')(alg_classifier)
alg_classifier = Dropout(.3)(alg_classifier)
alg_classifier = MaxPool2D(pool_size=(4,4), strides=4)(alg_classifier)
alg_classifier = BatchNormalization(axis = -1)(alg_classifier)
alg_classifier = Conv2D(alg_atten_dim, kernel_size=1)(alg_classifier)
alg_classifier = Reshape((-1, alg_atten_dim))(alg_classifier)
alg_classifier_attended = Normalization(axis = -1)(alg_classifier)
alg_classifier_attention = MultiHeadAttention(num_heads=alg_heads, key_dim=alg_key_dim)(alg_classifier_attended, alg_classifier_attended)
alg_classifier = Dropout(.2)(alg_classifier_attention)
alg_classifier = Flatten()(alg_classifier)
alg_classifier = Dense(len(algs), activation=activation)(alg_classifier)
alg_classifier = Model(alg_input, alg_classifier)

In [ ]:
alg_classifier.summary()

In [ ]:
#use sparse categorical entropy for loss function i think
#may need to resize?
enum_alg_targets, alg_mapping = make_mapping(train_targets, algs, 'alg')
print(enum_alg_targets)
alg_callback = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
alg_classifier.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
alg_hist = alg_classifier.fit(train_data_format, enum_alg_targets['alg'].to_numpy(), epochs=200, callbacks=[alg_callback])

In [ ]:
print(len(test_data.to_list()))

In [ ]:
'''did I overfit on test data with hyperparameter tuning?'''
'''.3 did best but most recent result not as good as .4 for dropout rate'''
enum_alg_test_targets, alg_mapping = make_mapping(test_targets, algs, 'alg')
alg_results = alg_classifier.evaluate(np.array(test_data.to_list()), np.array(enum_alg_test_targets['alg'].to_list()))
print(alg_results)

In [ ]:
'''Unsupervised Clustering'''
#make it more adaptive but rn 3 algs, 5 i think password, and some amount of plaintext
sc_algs = SpectralClustering(len(algs))
sc_pass = SpectralClustering(len(passwords))
sc_source = SpectralClustering(df['source_file'].n_unique())
array_input_train = np.array(df['ciphertext'].to_list())
flat_input_train = array_input_train.reshape(array_input_train.shape[0], -1)
print(flat_input_train)

In [ ]:
'''check performance and then do with the encoded training data'''
sc_alg_pred = sc_algs.fit_predict(flat_input_train)

In [ ]:
sc_pass_pred = sc_pass.fit_predict(flat_input_train)

In [ ]:
'''probably too many groups for spectral'''
sc_source = sc_source.fit_predict(flat_input_train)

In [ ]:
'''Supervised Confusion'''
#prolly same structure as alg classifier but with an output dimension to represent the amount of passwords
'''need to do one for each password maybe'''
'''could also just do it raw and see which algs get password guessed the most'''
pass_heads = 4
pass_key_dim =16
pass_atten_dim = pass_heads*pass_key_dim
pass_input = Input(shape=image_shape)
pass_classifier = Conv2D(max_blocks*4, (4,4), padding='same')(pass_input)
pass_classifier = BatchNormalization(axis = -1)(pass_classifier)
pass_classifier = Activation('relu')
pass_classifier = Dropout(.3)(pass_classifier)
pass_classifier = MaxPool2D(pool_size=(4,4), strides=4)(pass_classifier)
pass_classifier = BatchNormalization(axis = -1)(pass_classifier)
pass_classifier = Conv2D(max_blocks, (4,4), padding='same')(pass_classifier)
pass_classifier = BatchNormalization(axis = -1)(pass_classifier)
pass_classifier = Activation('relu')
pass_classifier = Dropout(.3)(pass_classifier)
pass_classifier = MaxPool2D(pool_size=(4,4), strides=4)(pass_classifier)
pass_classifier = BatchNormalization(axis = -1)(pass_classifier)
pass_classifier = Conv2D(pass_atten_dim, kernel_size=1)(pass_classifier)
pass_classifier = Reshape((-1, pass_atten_dim))(pass_classifier)
pass_classifier_attended = Normalization(axis = -1)(pass_classifier)
pass_classifier_attention = MultiHeadAttention(num_heads=pass_heads, key_dim=pass_key_dim)(pass_classifier_attended, pass_classifier_attended)
pass_classifier = Dropout(.2)(pass_classifier_attention)
pass_classifier = Flatten()(pass_classifier)
pass_classifier = Dense(len(passwords), activation='softmax')(pass_classifier)
pass_classifier = Model(pass_input, pass_classifier)

In [ ]:
enum_pass_targets, pass_mapping = make_mapping(train_targets, passwords, 'password')
pass_classifier.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
pass_callback = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
pass_hist = pass_classifier.fit(train_data_format, enum_pass_targets['password'].to_numpy(), epochs=200, callbacks=[pass_callback])

In [ ]:
enum_pass_test_targets, pass_mapping = make_mapping(test_targets, passwords, 'password')
pass_results = pass_classifier.evaluate(np.array(test_data.to_list()), np.array(enum_pass_test_targets['password'].to_list()))
print(pass_results)
'''highest so far was an accuracy of roughly .5, pretty good considering there are 5 passwords nvm mode collapse lol'''

In [ ]:
'''Supervised Diffusion'''
#same as above but with respect to the amount of sources
# encoder_input = Input(shape=image_shape)
# sourceclass1 = Conv2D(max_blocks*4, (4,4), padding='same', activation='relu')
# source_drop1 = Dropout(.2)
# sourceclass2 = MaxPool2D(pool_size=(4,4), strides=4)
# sourceclass3 = Conv2D(max_blocks, (4,4), padding='same', activation='relu')
# source_drop2 = Dropout(.2)
# sourceclass4 = MaxPool2D(pool_size=(4,4), strides=4)
# sourceclass5 = Flatten()
# source_prediction_layer = Dense(df['source_file'].n_unique(), activation='softmax')
# source_classifier = Sequential([encoder_input, sourceclass1, source_drop1, sourceclass2, sourceclass3, source_drop2, 
#                              sourceclass4, sourceclass5, source_prediction_layer])
source_heads = 4
source_key_dim =16
source_atten_dim = source_heads*source_key_dim
source_input = Input(shape=image_shape)
source_classifier = Conv2D(max_blocks*4, (4,4), padding='same')(source_input)
source_classifier = BatchNormalization(axis = -1)(source_classifier)
source_classifier = Activation('relu')
source_classifier = Dropout(.3)(source_classifier)
source_classifier = MaxPool2D(pool_size=(4,4), strides=4)(source_classifier)
source_classifier = BatchNormalization(axis = -1)(source_classifier)
source_classifier = Conv2D(max_blocks, (4,4), padding='same')(source_classifier)
source_classifier = BatchNormalization(axis = -1)(source_classifier)
source_classifier = Activation('relu')
source_classifier = Dropout(.3)(source_classifier)
source_classifier = MaxPool2D(pool_size=(4,4), strides=4)(source_classifier)
source_classifier = BatchNormalization(axis = -1)(source_classifier)
source_classifier = Conv2D(source_atten_dim, kernel_size=1)(source_classifier)
source_classifier = Reshape((-1, source_atten_dim))(source_classifier)
source_classifier_attended = Normalization(axis = -1)(source_classifier)
source_classifier_attention = MultiHeadAttention(num_heads=source_heads, key_dim=source_key_dim)(source_classifier_attended, source_classifier_attended)
source_classifier = Dropout(.2)(source_classifier_attention)
source_classifier = Flatten()(source_classifier)
source_classifier = Dense(df['source_file'].n_unique(), activation='softmax')(source_classifier)
source_classifier = Model(source_input, source_classifier)

In [ ]:
enum_source_targets, source_mapping = make_mapping(train_targets, sources, 'source_file')
source_classifier.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
source_callback = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
source_hist = pass_classifier.fit(np.array(train_data.to_list()), enum_source_targets['source_file'].to_numpy(), epochs=200, callbacks=[source_callback])

In [ ]:
enum_source_test_targets, source_mapping = make_mapping(test_targets, passwords, 'password')
source_results = pass_classifier.evaluate(np.array(test_data.to_list()), np.array(enum_pass_test_targets['password'].to_list()))
print(source_results)

In [ ]:
alg_probs = alg_classifier.predict(test_data_format)
alg_preds = np.array([np.argmax(prob) for prob in alg_probs])
print(alg_preds)
print(enum_alg_test_targets)
alg_inv_mapping = {idx: label for label, idx in alg_mapping.items()}
alg_true_label = [alg_inv_mapping[idx] for idx in enum_alg_test_targets['alg'].to_list()]
alg_pred_label = [alg_inv_mapping[idk] for idk in alg_preds]

In [ ]:
alg_confusion = confusion_matrix(alg_true_label, alg_pred_label, labels=algs)
alg_disp = ConfusionMatrixDisplay(confusion_matrix=alg_confusion, display_labels=algs)
alg_disp.plot(cmap="Blues", values_format = 'd')
plt.title("Alg Confusion")
plt.show()

In [ ]:
pass_probs = pass_classifier.predict(test_data_format)
pass_preds = np.array([np.argmax(prob) for prob in pass_probs])
print(pass_preds)
print(enum_pass_test_targets)
pass_inv_mapping = {idx: label for label, idx in pass_mapping.items()}
pass_true_label = [pass_inv_mapping[idx] for idx in enum_pass_test_targets['password'].to_list()]
pass_pred_label = [pass_inv_mapping[idk] for idk in pass_preds]

In [ ]:
pass_confusion = confusion_matrix(pass_true_label, pass_pred_label, labels=passwords)
pass_disp = ConfusionMatrixDisplay(confusion_matrix=pass_confusion, display_labels=passwords)
pass_disp.plot(cmap="Blues", values_format = 'd')
plt.title("Password Confusion")
plt.show()

In [ ]:
encoded_data = encoder.predict(np.array(df['ciphertext'].to_list()))

In [ ]:
alg_true, alg_map = make_mapping(df, algs, 'alg')
print(encoded_data[:, 0])
print(len(encoded_data[:, 0]))
print(len(encoded_data[:, 1]))
print(len(encoded_data[:, 2]))
print(len(alg_true))
print(len(sc_alg_pred))
print(alg_true.shape)
print(sc_alg_pred.shape)
alg_true = alg_true['alg']
print(np.unique(sc_alg_pred, return_counts = True))

In [ ]:
alg_cluster_results = pd.DataFrame({
    'X1': encoded_data[:, 0],
    'X2': encoded_data[:, 1],
    'X3': encoded_data[:, 2],
    'True Label': alg_true,
    'Predicted Cluster': sc_alg_pred
})

fig_true = px.scatter_3d(
    alg_cluster_results, x='X1', y='X2', z='X3',
    color='True Label',
    title='Alg True Labels in Autoencoded 3D Space'
)
fig_true.show()

fig_pred = px.scatter_3d(
    alg_cluster_results, x='X1', y='X2', z='X3',
    color='Predicted Cluster',
    title='Spectral Clustering in 3D Latent Space'
)
fig_pred.show()


In [ ]:
pass_true, pass_map = make_mapping(df, passwords, 'password')
print(encoded_data[:, 0])
print(len(encoded_data[:, 0]))
print(len(encoded_data[:, 1]))
print(len(encoded_data[:, 2]))
print(len(alg_true))
print(len(sc_pass_pred))
print(pass_true.shape)
print(sc_pass_pred.shape)
pass_true = pass_true['password']
print(np.unique(sc_pass_pred, return_counts = True))

In [ ]:
alg_cluster_results = pd.DataFrame({
    'X1': encoded_data[:, 0],
    'X2': encoded_data[:, 1],
    'X3': encoded_data[:, 2],
    'True Label': pass_true,
    'Predicted Cluster': sc_pass_pred
})

fig_true = px.scatter_3d(
    alg_cluster_results, x='X1', y='X2', z='X3',
    color='True Label',
    title='Password True Labels in Autoencoded 3D Space'
)
fig_true.show()

fig_pred = px.scatter_3d(
    alg_cluster_results, x='X1', y='X2', z='X3',
    color='Predicted Cluster',
    title='Spectral Clustering in 3D Latent Space'
)
fig_pred.show()


In [ ]:
alg_probs = alg_classifier.predict(test_data_format)
alg_preds = np.array([np.argmax(prob) for prob in alg_probs])
print(alg_preds)
print(enum_alg_test_targets)
alg_inv_mapping = {idx: label for label, idx in alg_mapping.items()}
alg_true_label = [alg_inv_mapping[idx] for idx in enum_alg_test_targets['alg'].to_list()]
alg_pred_label = [alg_inv_mapping[idk] for idk in alg_preds]

In [ ]:
alg_confusion = confusion_matrix(alg_true_label, alg_pred_label, labels=algs)
alg_disp = ConfusionMatrixDisplay(confusion_matrix=alg_confusion, display_labels=algs)
alg_disp.plot(cmap="Blues", values_format = 'd')
plt.title("Alg Confusion")
plt.show()